In [8]:
import requests
import pandas as pd
from dotenv import load_dotenv
import os
from datetime import datetime
import sqlite3
import time

load_dotenv()
API_KEY = os.getenv("API_KEY")
TOKEN = os.getenv("TOKEN")

print("APIs loaded successfully!")
print(f"API Key present: {'Yes' if API_KEY else 'No'}")
print(f"WAQI Token present: {'Yes' if TOKEN else 'No'}")

APIs loaded successfully!
API Key present: Yes
WAQI Token present: Yes


In [9]:
dataset = pd.read_csv("../Data/MainDataset.csv")

print("MainDataset loaded!")
print(f"Total records: {len(dataset)}")
print(f"Unique areas: {dataset['area'].nunique()}")

area_counts = dataset['area'].value_counts()
top_20_areas = area_counts.head(20).index.tolist()

print("\nTop 20 areas by record count:")
for i, area in enumerate(top_20_areas, 1):
    count = area_counts[area]
    print(f"{i:2}. {area:20} - {count:6,} records")

MainDataset loaded!
Total records: 425279
Unique areas: 295

Top 20 areas by record count:
 1. Delhi                -  3,679 records
 2. Hyderabad            -  3,668 records
 3. Bengaluru            -  3,656 records
 4. Chennai              -  3,611 records
 5. Aurangabad           -  3,603 records
 6. Lucknow              -  3,592 records
 7. Kanpur               -  3,515 records
 8. Navi Mumbai          -  3,500 records
 9. Agra                 -  3,458 records
10. Mumbai               -  3,420 records
11. Varanasi             -  3,408 records
12. Faridabad            -  3,379 records
13. Chandrapur           -  3,374 records
14. Jodhpur              -  3,272 records
15. Jaipur               -  3,255 records
16. Pune                 -  3,254 records
17. Muzaffarpur          -  3,163 records
18. Patna                -  3,125 records
19. Ahmedabad            -  3,073 records
20. Kolkata              -  3,046 records


In [10]:
print("="*60)
print("FETCHING WEATHER DATA FROM OPENWEATHERMAP API")
print("="*60 + "\n")

weather_data = []

for i, area in enumerate(top_20_areas, 1):
    print(f"[{i}/20] Fetching weather for {area}...", end=" ")
    
    try:
        url = f"http://api.openweathermap.org/data/2.5/weather?q={area}&appid={API_KEY}&units=metric"
        response = requests.get(url, timeout=5).json()
        
        if response.get("cod") == 200:
            main = response["main"]
            wind = response["wind"]
            
            weather_data.append({
                "area": area,
                "temperature": main.get("temp"),
                "humidity": main.get("humidity"),
                "pressure": main.get("pressure"),
                "wind_speed": wind.get("speed"),
                "wind_deg": wind.get("deg"),
                "cloud_pct": response["clouds"].get("all")
            })
            print("✓")
        else:
            print(f"✗ (Error: {response.get('message')})")
    
    except Exception as e:
        print(f"✗ (Exception: {str(e)[:40]})")
    
    time.sleep(0.5)

weather_df = pd.DataFrame(weather_data)
print(f"\n✓ Weather data fetched for {len(weather_df)}/{len(top_20_areas)} areas")
print("\nSample weather data:")
print(weather_df.head())

FETCHING WEATHER DATA FROM OPENWEATHERMAP API

[1/20] Fetching weather for Delhi... ✓
[2/20] Fetching weather for Hyderabad... ✓
[3/20] Fetching weather for Bengaluru... ✓
[4/20] Fetching weather for Chennai... ✓
[5/20] Fetching weather for Aurangabad... ✓
[6/20] Fetching weather for Lucknow... ✓
[7/20] Fetching weather for Kanpur... ✓
[8/20] Fetching weather for Navi Mumbai... ✓
[9/20] Fetching weather for Agra... ✓
[10/20] Fetching weather for Mumbai... ✓
[11/20] Fetching weather for Varanasi... ✓
[12/20] Fetching weather for Faridabad... ✓
[13/20] Fetching weather for Chandrapur... ✓
[14/20] Fetching weather for Jodhpur... ✓
[15/20] Fetching weather for Jaipur... ✓
[16/20] Fetching weather for Pune... ✓
[17/20] Fetching weather for Muzaffarpur... ✓
[18/20] Fetching weather for Patna... ✓
[19/20] Fetching weather for Ahmedabad... ✓
[20/20] Fetching weather for Kolkata... ✓

✓ Weather data fetched for 20/20 areas

Sample weather data:
         area  temperature  humidity  pressure  wi

In [11]:
print("="*60)
print("FETCHING AQI/AIR QUALITY DATA FROM WAQI API")
print("="*60 + "\n")

aqi_data = []

for i, area in enumerate(top_20_areas, 1):
    print(f"[{i}/20] Fetching AQI for {area}...", end=" ")
    
    try:
        url = f"https://api.waqi.info/feed/{area}/?token={TOKEN}"
        response = requests.get(url, timeout=5).json()
        
        if response["status"] == "ok":
            data = response["data"]
            iaqi = data.get("iaqi", {})
            
            aqi_data.append({
                "area": area,
                "aqi": data.get("aqi"),
                "pm25": iaqi.get("pm25", {}).get("v"),
                "pm10": iaqi.get("pm10", {}).get("v"),
                "no2": iaqi.get("no2", {}).get("v"),
                "so2": iaqi.get("so2", {}).get("v"),
                "co": iaqi.get("co", {}).get("v"),
                "o3": iaqi.get("o3", {}).get("v")
            })
            print("✓")
        else:
            print(f"✗ (Not found)")
    
    except Exception as e:
        print(f"✗ (Exception: {str(e)[:40]})")
    
    time.sleep(0.5)

aqi_df = pd.DataFrame(aqi_data)
print(f"\n✓ AQI data fetched for {len(aqi_df)}/{len(top_20_areas)} areas")
print("\nSample AQI data:")
print(aqi_df.head())

FETCHING AQI/AIR QUALITY DATA FROM WAQI API

[1/20] Fetching AQI for Delhi... ✓
[2/20] Fetching AQI for Hyderabad... ✓
[3/20] Fetching AQI for Bengaluru... ✓
[4/20] Fetching AQI for Chennai... ✓
[5/20] Fetching AQI for Aurangabad... ✓
[6/20] Fetching AQI for Lucknow... ✓
[7/20] Fetching AQI for Kanpur... ✓
[8/20] Fetching AQI for Navi Mumbai... ✓
[9/20] Fetching AQI for Agra... ✓
[10/20] Fetching AQI for Mumbai... ✓
[11/20] Fetching AQI for Varanasi... ✓
[12/20] Fetching AQI for Faridabad... ✓
[13/20] Fetching AQI for Chandrapur... ✓
[14/20] Fetching AQI for Jodhpur... ✓
[15/20] Fetching AQI for Jaipur... ✓
[16/20] Fetching AQI for Pune... ✓
[17/20] Fetching AQI for Muzaffarpur... ✓
[18/20] Fetching AQI for Patna... ✓
[19/20] Fetching AQI for Ahmedabad... ✓
[20/20] Fetching AQI for Kolkata... ✓

✓ AQI data fetched for 20/20 areas

Sample AQI data:
         area  aqi  pm25  pm10   no2  so2    co    o3
0       Delhi   97    97  82.0  25.9  4.3   5.7   6.2
1   Hyderabad   75    75  65.0  

In [12]:
print("="*60)
print("COMBINING AND PREPARING DATA")
print("="*60 + "\n")

api_data = pd.merge(weather_df, aqi_df, on="area", how="inner")
api_data["timestamp"] = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
api_data["date"] = datetime.now().strftime("%Y-%m-%d")

print(f"Total records combined: {len(api_data)}")
print(f"\nColumns: {api_data.columns.tolist()}")
print("\nCombined data sample:")
print(api_data.head())

COMBINING AND PREPARING DATA

Total records combined: 20

Columns: ['area', 'temperature', 'humidity', 'pressure', 'wind_speed', 'wind_deg', 'cloud_pct', 'aqi', 'pm25', 'pm10', 'no2', 'so2', 'co', 'o3', 'timestamp', 'date']

Combined data sample:
         area  temperature  humidity  pressure  wind_speed  wind_deg  \
0       Delhi        25.05        36      1007        1.03        30   
1   Hyderabad        27.73        61      1008        2.06       220   
2   Bengaluru        24.31        67      1010        2.57       120   
3     Chennai        27.49        87      1009        2.57       270   
4  Aurangabad        28.76        20      1009        5.86       340   

   cloud_pct  aqi  pm25  pm10   no2  so2    co    o3            timestamp  \
0          9   97    97  82.0  25.9  4.3   5.7   6.2  2026-04-14 04:36:33   
1         75   75    75  65.0   5.7  7.0   1.5  10.0  2026-04-14 04:36:33   
2         20  124   124  60.0   5.8  2.5   2.6  29.3  2026-04-14 04:36:33   
3         20

In [13]:
print("="*60)
print("SCHEDULE REGULAR UPDATES")
print("="*60 + "\n")

print("To schedule daily/weekly updates:")
print("\nOption 1: Windows Task Scheduler")
print("  - Create a batch file that runs this notebook")
print("  - Schedule it to run daily/weekly")
print("\nOption 2: Python APScheduler")
print("  - Install: pip install apscheduler")
print("  - Schedule this script to run periodically")
print("\nOption 3: Run manually")
print("  - Re-run this notebook whenever you need fresh data")

print("\n" + "="*60)
print("SUMMARY")
print("="*60)
print(f"\n✓ API Data Collection Complete!")
print(f"  • Top 20 areas processed")
print(f"  • Weather data: {len(weather_df)} areas")
print(f"  • AQI data: {len(aqi_df)} areas")
print(f"  • Combined records: {len(api_data)}")
print(f"  • Data saved to database")
print(f"  • Timestamp: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print("\n" + "="*60)

SCHEDULE REGULAR UPDATES

To schedule daily/weekly updates:

Option 1: Windows Task Scheduler
  - Create a batch file that runs this notebook
  - Schedule it to run daily/weekly

Option 2: Python APScheduler
  - Install: pip install apscheduler
  - Schedule this script to run periodically

Option 3: Run manually
  - Re-run this notebook whenever you need fresh data

SUMMARY

✓ API Data Collection Complete!
  • Top 20 areas processed
  • Weather data: 20 areas
  • AQI data: 20 areas
  • Combined records: 20
  • Data saved to database
  • Timestamp: 2026-04-14 04:36:33



In [14]:
print("="*60)
print("SAVING TO DATABASE")
print("="*60 + "\n")

db_path = "../Database/airwatch.db"

try:
    conn = sqlite3.connect(db_path)
    
    api_data.to_sql("api_data_log", conn, if_exists="append", index=False)
    
    print(f"✓ Data saved to database: {db_path}")
    print(f"✓ Records added: {len(api_data)}")
    
    total_records = pd.read_sql("SELECT COUNT(*) as count FROM api_data_log", conn)
    print(f"✓ Total records in database: {total_records['count'].iloc[0]}")
    
    conn.close()
    
except Exception as e:
    print(f"✗ Database error: {str(e)}")

print("\n" + "="*60)

SAVING TO DATABASE

✓ Data saved to database: ../Database/airwatch.db
✓ Records added: 20
✓ Total records in database: 40

